In [1]:
# --------------------------------------------------------
# 
# Postprocess results 
#
# --------------------------------------------------------

In [1]:
# --- Decide which CDR calculations to perform
cdr_calc_list = ["co2_flx",       
                 "camg_flx",
                 "totcat_flx",
                 # "carbalk_flx", # (turn this back on when the runs are re-run with fixed cflx file)
                 "rockdiss"]
# ---

In [ ]:
import fnmatch
import glob
import os
import pickle
import sys
import re
from typing import Tuple

import cmocean.cm as cmo
import fsspec         # for AWS integration
import s3fs
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D  # for custom legend entries (needed for contour plot)
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# --- read in cdr calculation functions  
from ew_workflows import cdr_fxns_postproc as cfp
# ---
#
# TODO: Update emissions calculator to handle time-varying rock application (!)
# 
#  ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# [ FOCUS RUNS ]
# exe_central1p5 ; exe_techBehaviorChange1p5 ; exe_noWeathering1p5 ; exe_lowCDR1p5
# ----------------------------------------------------------------
#
#
# --- add crop area options (it will save multiple identical clim dirs,
#     but rock-application-specific files will be added later)
repeat_clim_for_crop_area = False
crop_area_frac_arr = np.round(np.linspace(0.6, 1, 5), 2)    # crop_area_frac_arr = np.round(np.linspace(0.1, 1, 10), 2) # from 10% to 100% in 10% increments
# --- add fixed app rate options 
repeat_clim_for_apprate_fixed = True
app_rates_arr = np.array([5, 7.5, 10, 15, 20, 30, 40, 50])
if repeat_clim_for_crop_area and repeat_clim_for_apprate_fixed:
    print("Warning: Must select either crop area or constant apprate to repeat across for how this script is structured!")
# -------------------------------------------------------------------
#
#
# ************************************************
# --- [ settings ]
SAVE_GROUP = "gcam"
EXP_GROUP = "gcam_opr_psdfull_3axis1"
EXP_SET = 'exe_central1p5'
dustsp = "gbas" # TODO: update `emissions_calculator_df` to use the `dustsp` in the df if mineral is None
multiyear = False
runtype = "yearly_ltm"
extra_tag = "v1"
batchname = f"{EXP_GROUP}_{EXP_SET}_{dustsp}_{runtype}_{extra_tag}.csv"
runtype = "field"
time_horizon = 80.   # set the time_horizon for computing the integrated flux 
ctrl_conditions = {"dustrate": 0} 
collapse_cols = None #[ # [ list or None ] # params that are analyzed one at a time, rather than all combinations (we don't want an xr dimension for each param)
#     "dustrad", "secondary_min_rule", "poro_updated",
# ] 
if repeat_clim_for_crop_area:
    dfin_cols_to_keep = [   # become dimensions in later xr dataset
        "site", "crop_fraction" # (need dustrad for emissions calc)
    ] 
elif repeat_clim_for_apprate_fixed:
    dfin_cols_to_keep = [   # become dimensions in later xr dataset
        "site", "apprate_base", "dustrad", "cec_factor"   # for "*_psdfull_3axis1"
    ]     
dfin_cols_to_keep_time = dfin_cols_to_keep + ["time"]
save_results = True
savepath = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}"
# ************************************************


# --- read in the batch .csv
batchdir = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/batch/"
dfin = pd.read_csv(os.path.join(batchdir, batchname))

# ---
# groundwork
outdir = f"s3://carbonplan-carbon-removal/SCEPTER/scepter_output/{EXP_GROUP}/{EXP_SET}"

In [11]:
# --- add crop area column if asked (stripped from the aws path)
if repeat_clim_for_crop_area & ("crop_fraction" in dfin_cols_to_keep):
    dfin['crop_fraction'] = dfin['aws_bucket'].str.split('/').str[-1]
# --- add column for the full run ID
if multiyear:
    dfin["newrun_id_full"] = dfin["newrun_id"] + f"_composite_{runtype}"
else:
    dfin['newrun_id_full'] = dfin['newrun_id'] + "_" + dfin['dustsp'] + "_" + runtype + "_tau"+dfin["duration"].astype(float).astype(str).str.replace(".", "p")  # (duration has to be turned into float first because otherwise we miss the decimal pt)
# prepend aws_bucket subdir if runs are not all in the same, single outdir (need for proper path building bc we hardcoded a pass to a single outdir later)
if repeat_clim_for_crop_area & ("crop_fraction" in dfin_cols_to_keep):
    bucket_base = outdir.rstrip('/')
    bucket_suffix = dfin['aws_bucket'].str.rstrip('/').apply(
        lambda b: b[len(bucket_base) + 1:] if b.startswith(bucket_base + '/') else ''
    )
    mask = bucket_suffix != ''
    dfin.loc[mask, 'newrun_id_full'] = bucket_suffix[mask] + '/' + dfin.loc[mask, 'newrun_id_full']
if repeat_clim_for_apprate_fixed & ("apprate_base" in dfin_cols_to_keep):
    bucket_base = outdir.rstrip('/')
    bucket_suffix = dfin['aws_bucket'].str.rstrip('/').apply(
        lambda b: b[len(bucket_base) + 1:] if b.startswith(bucket_base + '/') else ''
    )
    mask = bucket_suffix != ''
    dfin.loc[mask, 'newrun_id_full'] = bucket_suffix[mask] + '/' + dfin.loc[mask, 'newrun_id_full']

# identify the control run(s)
dfin = cfp.find_control_runs(dfin, ctrl_conditions)
# add a column for the dustrate in ton_ha_yr
if "dustrate" in dfin.columns:
    dfin["dustrate_ton_ha_yr"] = dfin["dustrate"] / 100 

# add dustrad if we only have psdrain_meanRad
if "dustrad" not in dfin.columns:
    dfin['dustrad'] = dfin['psdrain_meanRad'] * 1e6 # [ convert m to um ]

# --- extract sites
sitenames = dfin['site'].unique()

# --- create the collapsed col
if collapse_cols:
    dfin["collapsed_inputs"] = dfin[collapse_cols].apply(
       lambda row: " | ".join(f"{col}={row[col]}" for col in collapse_cols),
        axis=1
    )
    # update the ds coords
    dfin_cols_to_keep = dfin_cols_to_keep + ["collapsed_inputs"]
    dfin_cols_to_keep_time = dfin_cols_to_keep_time + ["collapsed_inputs"]

# Flux postprocessing

In [12]:
# --- read in the postprocessed flux data 
# [ integrated flux ]
flx_dict_int = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type="int_flx", rockdiss_feedstock=None
)
# [ flux ]
flx_dict = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type="flx", rockdiss_feedstock=None
)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [13]:
# --- compute CDR fluxes (with losses considered)
# ---
# [ time-integrated ]
cdr_dict_int_full, cdr_dict_int_sum = cfp.cdr_int_per_group(
    flx_dict_int, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep,
    bysite = True
)
# [ transient ]
cdr_dict_full, cdr_dict_sum = cfp.cdr_int_per_group(
    flx_dict, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep,
    bysite = True
)

# **************************************************
if save_results:
    # create a dictionary of ds's to save 
    df_dict_save = {
        "cdr_dfs_int_sum.pkl": cdr_dict_int_sum,
        "cdr_dfs_int.pkl": cdr_dict_int_full,
        "cdr_dfs_sum.pkl": cdr_dict_sum,
        "cdr_dfs_full.pkl": cdr_dict_full,
    }
# **************************************************

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [14]:
# --- compute CDR w/ loss rates and convert to xr datasets
outds_int = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_sum, dims = dfin_cols_to_keep, 
    cdr_calc_list = cdr_calc_list, skip_loss=False, convert_time_to_timestep = False,
)
# --- repeat for time-dependent time-integrated data
outds_int_time = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True,
    convert_time_to_timestep = True,
)
# --- repeat for time-transient data (note dims now includes "time")
outds = cfp.cdr_ds(
    cdr_dict = cdr_dict_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True, 
    convert_time_to_timestep = True,
)

# **************************************************
if save_results:
    # create a dictionary of ds's to save 
    ds_dict = {
        "ds_cdr_time-integrated.zarr": outds_int,
        "ds_cdr.zarr": outds_int_time,
        "ds_cdr_transient.zarr": outds, 
    }
# **************************************************


solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


/Users/tylerkukla/Documents/GitHub/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1659: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Users/tylerkukla/Documents/GitHub/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1659: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


/Users/tylerkukla/Documents/GitHub/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1659: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Users/tylerkukla/Documents/GitHub/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1659: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout

### Optional: Add time-transient emissions

In [7]:
# # --- get emissions (over time if expand_over_time=True)
# # set the inputs ***************
# p80_input = 1.3e3
# Efactor_org = "MRO"
# # ******************************
# ds_emiss_int_time = cfp.emissions_calculator_ds(
#     outds_int_time,
#     dustsp,
#     time_horizon,
#     p80_input,
#     Efactor_org,
#     expand_over_time=True,
# )
# # **************************************************
# if save_results:
#     # create a dictionary of ds's to save 
#     ds_dict['ds_emiss.zarr'] = ds_emiss_int_time
# # **************************************************


In [15]:
# --- save results **************************************************************************
# 
if save_results:
    if ds_dict.get("ds_emiss.zarr"):
        # create the dictionary we'll save as the resource file
        savedict = cfp.create_save_dict_oneFS_emiss(
            EXP_SET, cdr_calc_list, dfin_cols_to_keep, collapse_cols, 
            batchname, multiyear, time_horizon, outdir, batchdir, p80_input,
            Efactor_org, EXP_GROUP=EXP_GROUP, 
        )
    else:
        # create the dictionary we'll save as the resource file
        savedict = cfp.create_save_dict_oneFS_noEmiss(
            EXP_SET, cdr_calc_list, dfin_cols_to_keep, collapse_cols, 
            batchname, multiyear, time_horizon, outdir, batchdir, 
        )
    
    # write the files
    fname_res = "_calc_inputs.res"  
    base_name = f"{EXP_GROUP}/{EXP_SET}"
    saved_here = cfp.save_batch_postproc_oneSet(
        savedict, savepath, base_name, fname_res, df_dict_save,
        ds_dict, save_directory=None, # (None means we make a new directory)
    )
# *********************************************************************************************    


Directory created: s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_opr_psdfull_3axis1/exe_central1p5_001


/Users/tylerkukla/Documents/GitHub/iam-rtm-compare/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


# Profile postprocessing

In [ ]:
# --- prepare the post-processing inputs
# 
batch_axes = dfin_cols_to_keep
if "dustrate_ton_ha_yr" not in batch_axes: # make sure we distinguish between case / ctrl
    batch_axes.append("dustrate_ton_ha_yr")
#
# dictionary for which files to process in the batch profile functions
proc_dict = {
    "adsorbed": True,
    "adsorbed_percCEC": True,
    "adsorbed_ppm": True,
    "aqueous": True,
    "aqueous_total": False,
    "bulksoil": True,
    "exchange_total": True,
    "gas": True,
    "rate": True,
    "soil_ph": True,
    "solid": True,
    "solid_sp_saturation": True, 
    "solid_volumePercent": True,
    "solid_weightPercent": True,
    "specific_surface_area": True,
    "surface_area": True,
}


: 

In [ ]:
# --- compile all the profile data into a single dataset
import warnings 
warnings.filterwarnings("ignore", category=FutureWarning) # ... kicking a can down the road

dsdict = cfp.prof_batchprocess_allvars(
    outdir, dustsp, dfin, batch_axes, proc_dict, 
    print_loop_updates = False
)

compiling profile data for gbas


### Save profile results

In [ ]:
cfp.save_batch_postproc_profOnly(
    dsdict, dustsp, save_directory = saved_here, 
    base_path=None, base_dir_name=None, use_zarr=True,
)

/Users/tylerkukla/Documents/GitHub/ew-workflows/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


's3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam_opr3_psdfull_25diam/exe_central1p5_001'

In [253]:
# ---